### Programming for Science and Finance

*Prof. Götz Pfeiffer, School of Mathematical and Statistical Sciences, University of Galway*

# Notebook 8: Numerical Methods

This notebook accompanies **Part II: Scientific Programming**. You will:

* learn how to approximate **derivatives** using finite differences and understand these formulas as convolutions
* implement the **trapezoidal rule** and relate it to cumulative sums
* use discrete convolution and smoothing kernels to interpret **numerical differentiation and integration**
* apply and compare **root-finding** methods, including bisection and Newton's method
* structure numerical algorithms using **classes** to encapsulate solver behavior
* solve simple **ordinary differential equations** with Euler's method

By the end of this notebook, you will have a unified view of numerical methods, in terms of discrete calculus and iterative schemes.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

## Task 1: Numerical Differentiation.

### Computing the Derivative of a Function.

* Recall the **definition of the derivative** of a function $f(x)$:
$$
f'(x) = \lim_{h \rightarrow 0} \frac{f(x+h) - f(x)}{h}.
$$
* Thus, for sufficiently small $h$, we get the following approximation of the derivative:
$$
f'(x) \approx \frac{f(x+h) - f(x)}{h}.
$$
* While it is not entirely clear what "sufficiently small" means, this is something we can implement in Python.
* As a function `diff` with the following parameters:  
  1. the function $f(x)$ that we wish to differentiate,
  2. the value of $h$,    
  returning a Python function that computes an **approximation** of the derivative $f'(x)$.

In [ ]:
def diff(f, h):
    def f1(x):
        return (f(x+h) - f(x))/h
    return f1

* Warning: if $h$ is too small, floating point rounding errors may dominate the result.
* Application to $f(x) = \frac12 x^2$ with derivative $f'(x) = x$ (and $h = 0.01$):

In [ ]:
f = lambda x : x**2/2
f1 = diff(f, 1/100)
[f1(x) for x in range(10)]

* The exact derivative is $f'(x) = x$, so every value is too big by about $0.005 = h/2$.  The error of the forward difference is proportional to $h$.

* Using a `lambda` expression, the python code for `diff` can be shortened to:

In [ ]:
def diff1(f, h):
    return lambda x: (f(x+h) - f(x))/h

In [ ]:
f = lambda x : x**2/2
f1 = diff1(f, 1/100)
[f1(x) for x in range(10)]

* Alternatively, the formula
  $$
  f'(x) = \lim_{h \to 0} \frac{f(x+h) - f(x-h)}{2h}
  $$
  can be used to approximate the derivative of $f$ as
  $$
  f'(x) \approx \frac{f(x+h) - f(x-h)}{2h}
  $$
  for "small" values of `h`.

* This can be implemented in a similar way:

In [ ]:
def diff2(f, h):
    return lambda x: (f(x+h) - f(x-h))/(2*h)

In [ ]:
f = lambda x : x**2/2
f1 = diff2(f, 1/100)
[f1(x) for x in range(10)]

* These values are correct to $12$ decimal places.  The central difference is **exact** for a quadratic, and in general its error is proportional to $h^2$ rather than $h$ -- a much better return for the same amount of work.

---
**Exercises.**

1. Write a function `diff3` that uses backward differences $(f(x) - f(x-h))/h$ to approximate
   the derivative of $f(x)$.

2. Plot and compare the effects of `diff1`, `diff2` and `diff3` on $f(x) = \frac12x^2$.

3. The warning above says that too small an $h$ lets rounding errors take over.  Use `diff2` to approximate $f'(1)$ for $f(x) = \frac12 x^2$ with $h = 10^{-k}$, $k = 1, 2, \dots, 15$, and find the $k$ that gives the best answer.  What happens beyond it?

---

## Task 2: Differentiation as Convolution.

* In a vectorized version, we often work with a function $f$ in the form of
  a **samples list** `yyy` of $N+1$ function values $y_k = f(x_k)$ for $k = 0, \dots, N$,
  where the $x_k$, $k = 0, \dots, N$ form a list `xxx` often constructed
  as
  ```python
  xxx = np.linspace(a, b, N+1)
  ```
  (Then the function can be plotted easily as `plt.plot(xxx, yyy)`, where `yyy = f(xxx)`.)  
* In such a context
  it is natural to set $h = (b-a)/N = x_{k+1} - x_k$, and the (second) formula for the 
  derivative of $f$ becomes
  $$
  f'(x_k) \approx \frac1{2h}(1 \cdot y_{k+1} + 0 \cdot y_k + (-1) \cdot y_{k-1}).
  $$

* Recall the convolution product $c = a * b$ of lists $a$ and $b$, where the $k$-th element of the product list $c$ is
  $$
  c_k = \sum_{i+j=k} a_i b_j
  = a_{-1} b_{k+1} + a_0 b_k + a_1 b_{k-1} + \dotsm.
  $$

* On comparison with the previous formula, one sees that **differentiation is convolution** of the samples list `yyy` with a kernel of the form $\frac1{2h}[1, 0 ,-1]$.  And note that $[1, 0, -1] = [1, 1] * [1, -1]$ is itself a convolution of a smoothing kernel $s = [1,1]$ and a differentiation kernel $d = [1, -1]$.  (In terms of polynomials, think: $(1 + x) (1 - x) = 1 - x^2$.)

* This yields a python function `diff_samples2` as follows.

In [ ]:
def diff_samples2(yyy, h):
    return np.convolve(yyy, np.array([1, 0, -1])/(2*h), "same")

* Apply this to $f(x) = \frac12 x^2$.

In [ ]:
f = lambda x: x**2/2
xxx = np.linspace(1, 10, 10)   # 10 sample points, at distance 1
h = xxx[1] - xxx[0]            # read the step size off the samples
yyy = f(xxx)
print(h)
diff_samples2(yyy, h)

* Not too bad, except for the endpoints: for this $f$, the central difference is exact in the interior.
* Note how the step size `h` is taken from the samples themselves.  Passing a value of `h` that does not match the spacing of `xxx` scales every derivative by the wrong factor -- a mistake that is easily made, and hard to see in a plot.

* The first differentiation formula could be implemented as follows:

In [ ]:
def diff_samples1(yyy, h):
    return np.convolve(yyy, np.array([1, -1])/h, "same")

In [ ]:
diff_samples1(yyy, h)

* Here every value is half a step **too small**: `[1, -1]/h` is a backward difference, which really estimates $f'$ at the midpoint $x - h/2$.  The central difference above has no such bias.

* For more flexibility, we can provide the width `w` of the smoothing kernel as optional parameter, while also fixing the two ends of the interval.

In [ ]:
def diff_samples(yyy, h, w=2):
    ker = np.convolve(np.ones(w), np.array([1, -1]))
    yy1 = np.convolve(yyy, ker/(w*h), "same")
    e = (w+1)//2   # replicate nearest valid value
    return np.pad(yy1[e:-e], e, "edge")

* Padding with `"edge"` replicates nearest valid values to avoid artificial oscillations at the boundaries.
* Let's apply this to $f(x) = \sin(2 \pi x) + \frac15 \cos(10 \pi x)$ for $x \in [0, 1]$.

In [ ]:
f = lambda x: np.sin(2*np.pi*x) + 0.2*np.cos(10*np.pi*x)
a, b = 0, 1
N = 500
xxx = np.linspace(a, b, N+1)
h = (b - a) / N

* We plot $10f(x)$ for the scale, and the approximated $f'(x)$ for 4 different values of `w`.

In [ ]:
yyy = f(xxx)
plt.plot(xxx, 10*yyy, label="10f")

for w in range(1, 5):
    plt.plot(xxx, diff_samples(yyy, h, w), label=f"w={w}")
_ = plt.legend()

---
**Exercises.**

1. Implement backward differences $f'(x) \approx (f(x) - f(x-h))/h$ as `diff_samples3` using a suitable convolution product.

2. Apply `diff_samples3` to $f(x) = \sin(2 \pi x) + \frac15 \cos(10 \pi x)$ for $x \in [0, 1]$,
   plot the result and compare it with the result of `diff_samples`.

3. `diff_samples` pads the sample values with `"edge"`.  Repeat the last plot with the default zero padding instead, and explain the spikes that appear at the two ends.

---

## Task 3: Numerical Integration.

* The **Riemann integral** $I = \int_a^b f(t)\, dt$ of a function $f$ over the interval $[a, b]$ is defined as a certain limit of weighted sums of function values.

* We can use that weighted sum as approximation to the integral:
  $$
  I \approx h \sum_{k=0}^N f(x_k),
  $$
  where, as above, $[a, b]$ is subdivided in $N$ intervals of length $h = \frac{b-a}{N}$.

* Alternatively, we can use the **Trapezoidal Rule**:
  $$
  I \approx \frac{b-a}{2N} (f(x_0) + 2f(x_1) + \dotsm + 2 f(x_{N-1}) + f(x_N))
  $$

* With this a function `trapez(f, N)` that takes a function `f` and a number `N` as input and constructs a function `F(a, b)` that can compute an approximation to the definite integral $\int_a^b f(t)\, dt$ can be implemented as follows.

In [ ]:
def trapez(f, N):
    def F(a, b):
        xxx = np.linspace(a, b, N+1)
        yyy = f(xxx)
        return (b-a)/(2*N) * (yyy[0] + yyy[-1] + 2*yyy[1:-1].sum())
    return F

* **Example.** $f(x) = 2x$ with $\int_a^b f(t)\, dt = b^2 - a^2$ and $N = 10$.
* Warning: choosing $N$ too small could result in a too coarse approximation.

In [ ]:
f = lambda x : 2*x
F = trapez(f, 10)
F(0, 5)

* Exactly $25 = 5^2 - 0^2$.  The trapezoidal rule replaces $f$ by a straight line on each subinterval, so for a linear $f$ it is exact, whatever the value of $N$.

### Integration as Convolution.

* Inspecting the formulas for integration one sees that **integration is convolution** of the samples list `yyy` with a kernel of the form $h [1, 1, \dots, 1]$, or of the form $\frac{h}{2}[1, 2, 2, \dots, 2, 1]$.  

* Note that $[1, 2, 2, \dots, 2, 1] = [1, 1] * [1, 1, \dots, 1]$, the convolution of a **short smoothing kernel** $[1,1]$ of width $w = 2$  and a **long summation kernel** $[1,1,\dots, 1]$(think: $(1 + x)(1 + x + x^2) = (1 + 2x + 2x^2 + x^3)$).

* Convolution with the long summation kernel $[1, 1, \dots, 1]$ is actually more efficiently
computed as a **cumulative sum**. That's what `np.cumsum()` does.

In [ ]:
l = [1, 2, 3, 4]
print(np.convolve(l, [1,1,1,1])[:len(l)])
print(np.cumsum(l))

* With these considerations in mind, we can implement integration by convolution
  with the width $w$ of the short smoothing kernel as an optional parameter:

In [ ]:
def integrate_samples(yyy, h, w=2):
    return np.cumsum((h/w) * np.convolve(yyy, np.ones(w)))[:len(yyy)]

* Note how we need to truncate the result to make it the same length as the samples array `yyy`. 
* Let's apply this to $f(x) = \sin(2 \pi x) + \frac15 \cos(10 \pi x)$ for $x \in [0, 1]$.

In [ ]:
f = lambda x: np.sin(2*np.pi*x) + 0.2*np.cos(10*np.pi*x)

a, b, N = 0, 1, 500
h = (b - a) / N

xxx = np.linspace(a, b, N+1)
yyy = f(xxx)

* Compute the antiderivative and plot the result.

In [ ]:
zzz = integrate_samples(yyy, h)
plt.plot(xxx, zzz, label="F")
_ = plt.legend()

* Now differentiate the antiderivative and compare the result with the original function $f$.

In [ ]:
plt.plot(xxx, diff_samples(zzz, h), label="F'(x)")
plt.plot(xxx, yyy, label="f(x)")
_ = plt.legend()

* Indistinguishable!  That's exactly what the **Fundamental Theorem of Calculus** (FTC) predicts.

* Looking at differentiation and integration as convolution, FTC takes this form: 
  $$
  [1,-1] * [1, 1, 1, \dotsc] = [1],
  $$
  i.e., the short difference kernel $[1, -1]$ and the infinite summation kernel $[1,1,1,\dots]$
  are each other's **convolution inverses**
  (think geometric series: $\frac{1}{1-x} = \sum_i x^i$).

---
**Exercises.**

1. Write a Python function `antiderivative(f, a, N)` that uses `trapez(f, N)` to approximate the antiderivative $F(x) = \int_a^x f(t)\,dt$ of $f(x)$.

2. Write a Python function `trapezoid_samples(yyy, h)` that first convolves `yyy` with
   the short smoothing kernel $k = \frac{h}2 [1, 1]$ and then returns the
   cumulative sum of the results as the list of antiderivative samples.  Compare the output
   with `integrate_samples(yyy, h)` for a test function.

3. Sample the function $f(x) = \sin(10x)+\frac{2}{5}\cos(20x)$ on the interval $x \in [0, \frac{\pi}{5}]$ using $N = 20$ subintervals.
  Compute the samples `yyy = f(xxx)` and the step size $h$.

4. Using `integrate_samples`, compute approximations to the antiderivative
   corresponding to window sizes $w = 1, 2, 3$.  Compute the exact antiderivative $F(x)$
   with $F(0) = 0$.  Plot all four curves, and discuss how they differ and why.

---

## Task 4: Root Finding.

A **root** (or **zero**) of a function $f \colon \mathbb{R} \to \mathbb{R}$
is a number $x \in \mathbb{R}$ such that $f(x) = 0$.  
Often, the equation $f(x) = 0$ cannot be solved for $x$.

A **Root Finding Algorithm** can approximate a root of $f$, without a guarantee to find **all** roots of $f$.  
There are several such algorithms, with varying degrees of sophistication.
We discuss two simple strategies.

### Bisection.

For a given **tolerance** $\epsilon$, and a maximum number $M$ of iterations,

* **Initially**, pick an interval $[a, b]$ such that $f(a) \cdot f(b) < 0$, i.e., $f(a)$ and $f(b)$ have **opposite signs**.  (Then by the [IVT](https://en.wikipedia.org/wiki/Intermediate_value_theorem), there exists at least one $x \in (a, b)$ with $f(x) = 0$.)

* **Repeat** at most $M$ times:
  * $c = \frac{a + b}{2}$ (midpoint)
  * replace either $a$ or $b$ by $c$ so that $f(a) \cdot f(b) < 0$

  until $|f(c)| < \epsilon$  
* **Return** $c$ as an approximation to a root of $f$

This method is simple but slow, as it needs approximately $\log_2 \frac{b-a}{\epsilon}$ steps.

### Newton's Method.

The [Newton-Raphson Method](https://en.wikipedia.org/wiki/Newton%27s_method) uses the derivative $f'(x)$ to locate a zero of $f(x)$. 

For a given **tolerance** $\epsilon$, and a maximum number $M$ of iterations,

* **Initially**, guess a value $x$.

* **Repeat** at most $M$ times:
  * replace $x$ by $x - f(x)/f'(x)$

  until $|f(x)| < \epsilon$

* **Return** $x$ as an approximation to a root of $f$.

This method converges quadratically, if the initial guess $x$ is close enough to the actual root.

---
**Exercises.**

1. Implement the bisection method as a Python function `bisection`.

2. Apply your `bisection` function to solve $x^2 = 2$.

3. Implement the Newton method as a Python function `newton`.

4. Apply your `newton` function to solve $x^2 = 3$.

---

## Task 5: A Root Finding Class.

Seeing the above two root finding methods have a lot of structure in common, it might be a good idea to implement both (and perhaps other root finding methods) with a common API.

Both methods work with more or less the same setup, consisting of:

* a function $f(x)$,
* its derivative $f'(x)$ (optional)
* a tolerance $\epsilon$ (default value $10^{-6}$)
* a maximal number $M$ of iterations (default value $99$).

With this setup, one can build a function `solve` that finds a root, starting with one or two initial values.

For an implementation, we choose the following **design**. We define an abstract parent **class**
`RootFinder` for the general setup.  Different strategies can then be defined as **subclasses** of `RootFinder` and only need to implement the `solve` function.

In [ ]:
class RootFinder:
    def __init__(self, f, f1=None, eps=1e-6, M=99):
        self.f = f
        self.f1 = f1
        self.eps = eps
        self.M = M

    def solve(self, a, b= None):
        raise NotImplementedError("Subclass must implement solve().")

The `solve` method in the parent class serves only as a placeholder: it raises a `NotImplementedError` (Notebook 4, Task 8), so that a subclass which forgets to provide its own `solve` fails with a clear message.  The subclasses below apply the same idea to their own preconditions: each raises a `ValueError` when it is given arguments it cannot work with -- an interval whose end points do not bracket a root, or a missing derivative.

The Bisection method as a subclass `BisectionSolver` inherits the `__init__` method and
hence needs no implementation of its own.

In [ ]:
class BisectionSolver(RootFinder):
    def solve(self, a, b):  # overwrite parent's method
        f = self.f
        if f(a) * f(b) >= 0:
            raise ValueError("end points must have opposite signs")
        for i in range(self.M):
            c = (a+b)/2
            if abs(f(c)) < self.eps:
                return c
            if f(a) * f(c) < 0:
                b = c
            else:
                a = c
        print("Warning: maximum iterations reached.")
        return c

Application to $x^2 = 2$, i.e., find $\sqrt{2}$:

* $f(x) = x^2 - 2$.
* $[a, b] = [1, 2]$.

In [ ]:
solver = BisectionSolver(lambda x: x**2 - 2)
x = solver.solve(1, 2)
print(x, x**2)

The Newton method as a subclass `NewtonSolver` inherits the `__init__` method and
hence needs no implementation of its own.

In [ ]:
class NewtonSolver(RootFinder):
    def solve(self, a, b=None):
        f, f1 = self.f, self.f1
        if f1 is None:
            raise ValueError("Newton's method needs the derivative f1")
        for i in range(self.M):
            y = f(a)
            if abs(y) < self.eps:
                return a
            a -= y/f1(a)
        print("Warning: maximum iterations reached.")
        return a

Application to $x^2 = 3$, i.e., find $\sqrt{3}$:

* $f(x) = x^2 - 3$
* $f'(x) = 2x$
* $a = 1$.

In [ ]:
solver = NewtonSolver(lambda x: x**2 - 3, lambda x: 2*x)
x = solver.solve(1)
print(x, x**2)

Both solvers find their root, but not with the same effort: bisection halves the interval each time and needed $21$ steps for $\sqrt2$, while Newton reached $\sqrt3$ to $9$ digits in $5$.

### Kepler's Equation.

Root finding earns its keep on equations that cannot be solved in closed form at all.  The
oldest and most famous of these is **Kepler's equation**.  A planet moves on an ellipse of
eccentricity $e$, and its position at a given time is described by the **eccentric anomaly**
$E$, an angle.  Time enters through the **mean anomaly** $M$, which simply increases at a
constant rate through the orbit.  The two are related by
$$
  M = E - e \sin E .
$$
Given the time, that is given $M$, we want $E$ -- and the equation cannot be rearranged for
it.  Every ephemeris, and every spacecraft trajectory, solves it numerically.

Written as a root finding problem, $f(E) = E - e \sin E - M$, with derivative
$f'(E) = 1 - e \cos E$.  Here it is for Mercury, whose orbit is the most eccentric of the
planets, a quarter of the way through its year.

In [ ]:
e = 0.2056                  # eccentricity of Mercury's orbit
M = 0.5                     # mean anomaly, in radians

f  = lambda E: E - e * math.sin(E) - M
f1 = lambda E: 1 - e * math.cos(E)

E = NewtonSolver(f, f1).solve(M)          # start the search at E = M
print("eccentric anomaly:", E)
print("check:", E - e * math.sin(E))

The check returns $0.5$, to within the solver's tolerance.  Note the initial guess: $E = M$ is the answer for a
circular orbit ($e = 0$), and so a good starting point for an orbit that is nearly circular.
This is why Newton's method is the standard choice here -- with a sensible guess it converges
in a handful of steps, and the equation has to be solved over and over again.

---
**Exercises.**

1. Find a root of $f(x) = x^3 - x - 2$ between $1$ and $2$.

2. Solve $\cos x = x$ and $x e^x = 2$, both with $[a, b] = [0, 1]$.

3. Implement a Newton solver that computes $f'(x)$ from $f(x)$ rather than taking it as an explicit argument.

4. Modify the solver classes by renaming `solve` to `__call__` (recall **Callable Objects** in Notebook 4).  Why would that be a good idea?

5. Solve Kepler's equation for Mercury ($e = 0.2056$) at $M = 0.1, 0.2, \dots, 3.0$, and plot $E$ against $M$.  Where does the curve depart furthest from the straight line $E = M$, and what does that say about the planet's speed along its orbit?

---


## Task 6: Solving Differential Equations.

A **first order initial value problem** (IVP)  has the form
$$
y'(t) = f(t, y(t)),\\
y(t_0) = y_0,
$$
i.e., a first order **differential equation** (expressing the derivative of a function $y(t)$ in terms of $y$ and $t$) and an **initial condition** (specifying the value of $y$ at a particular point).   The goal is to find a function $y(t)$ that satisfies both the differential equation and the initial condition.

### Euler's Method.

A simple method for approximating a solution to an IVP uses the approximation to the derivative
$$
y'(t) \approx \frac{y(t + h) - y(t)}{h}
$$
Setting
$$
t_n = t_{n-1} + h,\quad n > 0,
$$
and
$$
y_n = y(t_n),
$$
we can rewrite
$$
f(t_0, y_0) = y'(t_0) \approx \frac{y(t_1) - y(t_0)}{h}
= \frac{y_1 - y_0}{h}
$$
as an approximation
$$
y_1 \approx y_0 + h \cdot f(t_0, y_0)
$$
and then, continuing in that fashion:
$$
y_{n+1} \approx y_n + h \cdot f(t_n, y_n).
$$

This process can be implemented as follows.

In [ ]:
def euler(f, t0, y0, h):
    def approx(n):
        t, y = t0, y0
        for i in range(n):
            y += h * f(t, y)
            t += h
        return y
    return approx

Note that `euler` does not return an approximation, it returns a **function**: the inner `approx` is a **closure** (Notebook 7) which has captured `f`, `t0`, `y0` and `h`, and which computes $y_n$ when it is called with a number `n` of steps.  So one call of `euler` sets up the method, and the function it hands back can then be applied as often as needed, for different values of `n`.

**Example.** Consider
$$
y' = 2y +t^2 - t \\ y(0) = 1.
$$
with $h = 0.01$, say.

An exact solution to this IVP is $y = e^{2t} - t^2/2$, as can be easily verified:
clearly, $y(0) = 1$, and $y' = 2e^{2t} - t$.

In [ ]:
e = euler(lambda t, y: 2*y + t**2 - t, 0, 1, 0.01)
e(100)

Plot the estimate and the exact solution:

In [ ]:
ttt = np.linspace(0, 1, 101)
y = lambda t: np.exp(2*t) - t**2/2  # exact solution
plt.plot(ttt, y(ttt), label="exact")
plt.plot(ttt, [e(n) for n in range(101)], label="estimate")
_ = plt.legend()

The exact value is $y(1) = e^2 - \tfrac12 = 6.8891$, so the estimate $6.7603$ is about $2\%$ low.  In the plot the two curves drift apart as $t$ grows: each Euler step follows the tangent at the **left** end of its interval, which for a curve bending upwards always falls short.

---
**Exercises.**

1. Solve $y' = -2y$, $y(0) = 100$ (with $h = 1$ and $N = 100$) and compare the result with the exact solution $y(t) = 100 e^{-2t}$.

2. Repeat the example of this task with $h = 0.1$, $0.01$ and $0.001$, and record the error at $t = 1$ in each case.  Roughly how does the error shrink as $h$ does?

3. In the example above, Euler's method **underestimates** the exact solution throughout.  Looking at the plot, explain why, in terms of the curvature of the solution.  Does the same happen for $y' = -2y$?

---


## Task 7: Numerical Libraries.

The algorithms in this notebook were built from scratch, in order to see how they work.  In practice one uses the library **`scipy`**, which builds on `numpy` and provides tested, efficient and much more sophisticated versions of all of them:

* `scipy.optimize.brentq(f, a, b)` finds a root of `f` in the interval $[a, b]$, and `scipy.optimize.newton` implements Newton's method;
* `scipy.integrate.quad(f, a, b)` computes a definite integral, and `scipy.integrate.solve_ivp` solves initial value problems;
* `scipy.special` contains the special functions of mathematics -- Notebook 9 will use `scipy.special.erf` for the normal distribution.

Knowing what these functions do, and where they can go wrong, is exactly what the hand-made versions above are good for.

In [ ]:
import scipy
print(scipy.__version__)
from scipy.optimize import brentq
from scipy.integrate import quad

print(brentq(lambda x: x**2 - 2, 1, 2))   # compare: BisectionSolver
print(quad(lambda x: 2*x, 0, 5))          # value, and error estimate

`brentq` agrees with our `BisectionSolver` to $7$ decimal places, and is itself correct to more than $12$.  Note also that `quad` returns a **pair**: the value of the integral, and an estimate of its own error.

---
**Exercises.**

1. Use `scipy.integrate.quad` to compute $\int_0^1 e^{-x^2}\,dx$, and compare with the `trapez` function of Task 3.  How many subintervals does `trapez` need in order to agree with `quad` to $8$ decimal places?

2. Solve the initial value problem of Task 6 with `scipy.integrate.solve_ivp`, and compare its value at $t = 1$ with the one from `euler`.

3. `scipy.optimize.newton` accepts the derivative as an optional argument `fprime`.  Solve $x^2 = 3$ with it and without it, and compare both with the `NewtonSolver` of Task 5.

---

## Summary.

* introduced **finite difference methods** for numerical differentiation
* interpreted differentiation as **convolution with short kernels**
* implemented **numerical integration**, including the trapezoidal rule
* connected integration to **cumulative sums and convolution**
* applied two core **root-finding** methods: bisection and Newton’s method
* organized solvers into **classes** to encapsulate algorithms
* solved simple **ordinary differential equations** using Euler’s method
* highlighted how differentiation, integration, and iteration form the basis of numerical computation